# RAPiDock Colab Runner (Google Drive mode)

This notebook mounts Google Drive and reads/writes files directly from your Drive copy of the project.

Expected project location in Drive:
- `/content/drive/MyDrive/ghsr_af3_molecule_screening`

You can change the path in the configuration cell if needed.

**Important:** RAPiDock ships precompiled Linux modules (for example `PeptideBuilder.so`) that are **not compatible with Colab's default Python 3.12** (you may see `undefined symbol: _PyUnicode_Ready`). This notebook installs a **conda Python 3.10** environment and runs `inference.py` via `conda run`.

In [1]:
#@title 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# If Drive is already mounted from a previous cell in this session, you can skip this cell.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title 2) Configure project paths on Drive
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ghsr_af3_molecule_screening')  # change if needed
RAPIDOCK_REPO = Path('/content/RAPiDock')
CONDA_ENV = 'rapidock310'

RECEPTOR_PDB = PROJECT_DIR / 'data' / 'ghsr_receptor.pdb'
BATCH_CSV = PROJECT_DIR / 'inputs_rapidock' / 'protein_peptide.csv'
OUTPUT_SINGLE = PROJECT_DIR / 'outputs_rapidock_test'
OUTPUT_BATCH = PROJECT_DIR / 'outputs_rapidock'

# Make paths visible to subsequent `!shell` cells
for _k, _p in {
    'RECEPTOR_PDB': RECEPTOR_PDB,
    'BATCH_CSV': BATCH_CSV,
    'OUTPUT_SINGLE': OUTPUT_SINGLE,
    'OUTPUT_BATCH': OUTPUT_BATCH,
}.items():
    os.environ[_k] = str(_p)

print('PROJECT_DIR:', PROJECT_DIR)
print('CONDA_ENV:', CONDA_ENV)
print('RECEPTOR_PDB exists:', RECEPTOR_PDB.exists())
print('BATCH_CSV exists:', BATCH_CSV.exists())

In [ ]:
#@title 3) Install Miniforge + Python 3.10 env (RAPiDock `.so` requires Python ≤3.11)
# Colab's default Python is 3.12+; bundled PeptideBuilder.so links symbols removed in 3.12.

import os
import subprocess
from pathlib import Path

CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')
MAMBA_ROOT = Path('/usr/local/miniforge3')
CONDA = MAMBA_ROOT / 'bin' / 'conda'

if not CONDA.exists():
    installer = Path('/tmp/Miniforge3.sh')
    url = 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'
    subprocess.check_call(['wget', '-q', '-O', str(installer), url])
    subprocess.check_call(['bash', str(installer), '-b', '-p', str(MAMBA_ROOT)])

exists = subprocess.run(
    [str(CONDA), 'env', 'list'],
    check=True,
    capture_output=True,
    text=True,
).stdout
if f'{CONDA_ENV} ' not in exists:
    subprocess.check_call([str(CONDA), 'create', '-y', '-n', CONDA_ENV, 'python=3.10'])

print('conda env ready:', CONDA_ENV)
subprocess.check_call([str(CONDA), 'run', '-n', CONDA_ENV, 'python', '-V'])

In [ ]:
#@title 4) Clone RAPiDock
!git clone https://github.com/huifengzhao/RAPiDock.git
%cd /content/RAPiDock

In [9]:
#@title 5) Install RAPiDock dependencies inside the conda env (torch+pyg matched)
import os
import subprocess

import torch

CONDA = '/usr/local/miniforge3/bin/conda'
CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')

subprocess.check_call([CONDA, 'run', '-n', CONDA_ENV, 'python', '-m', 'pip', 'install', '-U', 'pip'])

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyyaml',
        'pandas',
        'biopython',
        'MDAnalysis>=2.9.0',
        'fair-esm',
        'e3nn',
        'pyrosetta-installer',
    ]
)

subprocess.check_call(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-m', 'pip', 'install', 'rdkit']
)

ver = torch.__version__
if '+' in ver:
    torch_base, pt_cuda = ver.split('+', 1)
else:
    torch_base, pt_cuda = ver, 'cpu'

if pt_cuda == 'cpu':
    torch_index = 'https://download.pytorch.org/whl/cpu'
else:
    torch_index = f'https://download.pytorch.org/whl/{pt_cuda}'

print('Notebook torch (for CUDA wheel selection):', ver)
print('Installing conda-env torch from:', torch_index)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        f'torch=={torch_base}',
        '--index-url',
        torch_index,
    ]
)

probe = subprocess.check_output(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-c', 'import torch; print(torch.__version__)'],
    text=True,
).strip()
if '+' in probe:
    env_base, env_cuda = probe.split('+', 1)
else:
    env_base, env_cuda = probe, 'cpu'
pyg_tag = f'cu{env_cuda.replace("cu", "")}' if env_cuda.startswith('cu') else 'cpu'
pyg_url = f'https://data.pyg.org/whl/torch-{env_base}+{pyg_tag}.html'

subprocess.run(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'uninstall',
        '-y',
        'torch-geometric',
        'torch-scatter',
        'torch-sparse',
        'torch-cluster',
        'pyg-lib',
    ],
    check=False,
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyg_lib',
        'torch_scatter',
        'torch_sparse',
        'torch_cluster',
        'torch_geometric',
        '-f',
        pyg_url,
    ]
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-c',
        'import MDAnalysis, rdkit, torch, torch_geometric, torch_scatter, torch_sparse, torch_cluster; '
        'print("torch:", torch.__version__); '
        'print("MDAnalysis:", MDAnalysis.__version__); '
        'print("RDKit:", rdkit.__version__); '
        'print("torch_geometric:", torch_geometric.__version__)',
    ]
)

SyntaxError: invalid syntax (3570215952.py, line 13)

In [6]:
#@title 6) Download checkpoint
!mkdir -p /content/RAPiDock/train_models/CGTensorProductEquivariantModel
!wget -O /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt "https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1"
!ls -lh /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt

--2026-04-30 02:53:26--  https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 137.138.52.235, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 56648773 (54M) [application/octet-stream]
Saving to: ‘/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt’

/content/RAPiDock/t 100%[===================>]  54.02M  19.7MB/s    in 2.7s    

2026-04-30 02:53:29 (19.7 MB/s) - ‘/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt’ saved [56648773/56648773]

-rw-r--r-- 1 root root 55M Apr 30 02:53 /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt


In [7]:
#@title 7) Single-case smoke test to Drive output
!conda run --no-capture-output -n "$CONDA_ENV" python /content/RAPiDock/inference.py \
  --complex_name hexarelin_test \
  --protein_description "$RECEPTOR_PDB" \
  --peptide_description "H[DTR]AW[DPN]K" \
  --output_dir "$OUTPUT_SINGLE" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 1 --batch_size 1 \
  --no_final_step_noise \
  --inference_steps 8 --actual_steps 8 \
  --conformation_partial 1:1:1 \
  --cpu 4

Traceback (most recent call last):
  File "/content/RAPiDock/inference.py", line 13, in <module>
    import MDAnalysis
ModuleNotFoundError: No module named 'MDAnalysis'


In [ ]:
#@title 8) Full batch run to Drive output (optional)
!conda run --no-capture-output -n "$CONDA_ENV" python /content/RAPiDock/inference.py \
  --protein_peptide_csv "$BATCH_CSV" \
  --output_dir "$OUTPUT_BATCH" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 10 --batch_size 4 \
  --no_final_step_noise \
  --inference_steps 16 --actual_steps 16 \
  --conformation_partial 1:1:1 \
  --cpu 8

In [ ]:
#@title 9) Verify Drive outputs
!ls -R "$OUTPUT_SINGLE"
print('---')
!ls -R "$OUTPUT_BATCH"

## Next step (local parsing)

After Colab finishes, go back to your local repo and run:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

Because outputs are written directly to Drive, no manual download/upload step is needed.